In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = ['C29', 'C37', 'C38', 'C39']
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2','IPS']

In [ ]:
leiden_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/integrate_m3c/leiden_clusters_renamed.csv'
leiden = pd.read_csv(leiden_path, sep='\t', header=0, index_col=0)
leiden['donor_leiden'] = [f'{x}_{y}' for x,y in zip(leiden['line'], leiden['renamed_leiden'])]
leiden.shape

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
clusters['donor_leiden'] = leiden.reindex(clusters.index)['donor_leiden']
tmp_clusters = clusters[~clusters['donor_leiden'].isna()].copy()
tmp_clusters['donor_leiden'] = tmp_clusters['donor_leiden'].replace('C39_Inter_2', 'C39_Fibro_15')
tmp_clusters.shape

In [ ]:
metas = []
for d in donors:
    donor_clusters = tmp_clusters[tmp_clusters['donor'] == d]
    tmp_clusts = sorted(donor_clusters['donor_leiden'].unique())
    tmp_meta = pd.DataFrame(index=tmp_clusts, columns=tmp_order)
    for s in tmp_order:
        final_clusters = donor_clusters[donor_clusters['pred_coarse'] == s]
        tmp_meta[s] = final_clusters['donor_leiden'].value_counts()
    tmp_meta = tmp_meta.div(tmp_meta.sum(axis=1), axis=0)
    tmp_meta['donor'] = d
    metas.append(tmp_meta)

In [ ]:
merged = pd.concat(metas).fillna(0)
merged.index = ['_'.join(x.split('_')[1:]) for x in merged.index]
merged.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for i,ax in enumerate(axes.flatten()):
    d = donors[i]
    to_plot = merged[merged['donor'] == d].iloc[:,:-1]
    sns.heatmap(to_plot, cmap='coolwarm', ax=ax)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2/'
merged.to_csv(f'{outdir}/snmct_rna_mc_cluster_props.csv', sep='\t')

### Inter2

In [ ]:
select = ['Inter2', 'donor']
tmp_merged = merged[select].copy()
tmp_merged.index.name = 'cluster'
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2/'
tmp_merged.to_csv(f'{outdir}/snmct_fibro_inter2_props.csv', sep='\t')

### timepoint

In [ ]:
tmp_times = sorted(clusters['time_num'].unique())

In [ ]:
metas = []
for d in donors:
    donor_clusters = tmp_clusters[tmp_clusters['donor'] == d]
    tmp_clusts = sorted(donor_clusters['donor_leiden'].unique())
    tmp_meta = pd.DataFrame(index=tmp_clusts, columns=tmp_times)
    for s in tmp_times:
        final_clusters = donor_clusters[donor_clusters['time_num'] == s]
        tmp_meta[s] = final_clusters['donor_leiden'].value_counts()
    tmp_meta = tmp_meta.div(tmp_meta.sum(axis=1), axis=0)
    tmp_meta['donor'] = d
    metas.append(tmp_meta)
    
merged = pd.concat(metas).fillna(0)
merged.index = ['_'.join(x.split('_')[1:]) for x in merged.index]
merged.index.name = 'cluster'
merged.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for i,ax in enumerate(axes.flatten()):
    d = donors[i]
    to_plot = merged[merged['donor'] == d].iloc[:,:-1]
    sns.heatmap(to_plot, cmap='coolwarm', ax=ax)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
merged.to_csv(f'{outdir}/snmct_mc_cluster_time_props.csv', sep='\t')

### plot Inter2

In [ ]:
!date